<a href="https://colab.research.google.com/github/SHREYA-G-AMIN/Agentic-AI/blob/main/Day_06_03_multi_agent_crewai_(3).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  AI Topic Summarizer using CrewAI (Multi-Agent Workflow)

Welcome to this project, where we demonstrate the power of **multi-agent collaboration** using [CrewAI](https://www.crewai.com/). In this notebook, you'll see how a team of AI agents — powered by a Language Model (LLM) — can work together to research and summarize any AI topic in a structured, exam-friendly format.

## Project Objective

To build a smart and modular system that:
- Accepts an **AI topic** from the user
- Uses one agent to **research** the topic
- Uses another agent to **summarize** the research into clean, indexed points
- Optionally uses a third agent to **validate** or **refine** the output

Why Multi-Agent?

Instead of using one AI for everything, we split the workload:
- Each agent specializes in a specific task
- Tasks are clearer and outputs are more structured
- Mimics real-world teamwork — where one person researches, another writes, another edits

##  Install CrewAI Library

We start by installing the `crewai` library along with optional tools.  
This library helps us build collaborative agents powered by Language Models.

>  This step only needs to be run once (and may already be pre-installed in some environments).


In [ ]:
%pip install -q crewai[tools]

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 6.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.4/202.4 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 841.4/841.4 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 73.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.4/269.4 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48

##  Import Required Modules

We import the core classes from **CrewAI**:

- `LLM`: The language model to be used by agents
- `Agent`: Defines individual AI agents
- `Task`: Describes what each agent must do
- `Crew`: Manages the coordination between agents

We also import Python's built-in `textwrap` module to help format long text for better readability.


In [ ]:
from crewai import LLM, Agent, Task, Crew
import textwrap

##  Set Up the Language Model

We configure the LLM (`gpt-4.1-nano`) using Nexus API.  
This model will power all our agents.


In [ ]:
llm = LLM(
    model="gpt-4.1-nano",
    temperature=0.7,
    base_url="https://nexusapi.navigatelabs.ai",
    api_key="sk-6Zx30euysY75UaRNfQyXJg")

##  Define the AI Agents

We create three agents, each with a specific role:
- `research_agent`: Gathers key facts about the topic
- `summarizer_agent`: Writes a clear, structured summary
- `tester_agent`: Reviews and improves the summary


In [ ]:
research_agent = Agent(
    role="AI Research Analyst",
    goal="Research and extract the most important facts about a given AI topic",
    backstory="An expert in AI who breaks down topics into core components for summarization.",
    tools=[],
    llm=llm,
    verbose=True
)

# Summary Generator Agent
summarizer_agent = Agent(
    role="Summary Generator",
    goal="Write clear, structured summaries based on researched content",
    backstory="An assistant skilled at creating exam-ready indexed summaries from detailed content.",
    tools=[],
    llm=llm,
    verbose=True
)

# Summary Tester Agent
tester_agent = Agent(
    role="Summary Tester",
    goal="Verify and improve the summary for clarity, structure, and exam effectiveness",
    backstory="An expert reviewer who checks AI summaries for completeness, accuracy, and readability.",
    tools=[],
    llm=llm,
    verbose=True
)

## Get User Input

Ask the user to enter the AI topic to be summarized.


In [ ]:
topic = input("Enter the AI topic you want summarized: ")

Enter the AI topic you want summarized: Elemental


##  Define the Tasks for Each Agent

- `research_task`: Collects detailed info about the topic  
- `summary_task`: Converts research into a clean, indexed summary  
- `testing_task`: Reviews and enhances the summary for clarity and completeness

In [ ]:
research_task = Task(
    description=f"Research and extract the key concepts, definitions, and examples for the topic: '{topic}'",
    expected_output="A structured research note covering key concepts, definitions, and any relevant subtopics.",
    agent=research_agent
)

# Task 2: Create summary from research
summary_task = Task(
    description="Using the research notes, generate a concise exam-style summary with 5–7 indexed bullet points.",
    expected_output="A summary in the format: 1. ..., 2. ..., 3. ..., covering all important points clearly.",
    agent=summarizer_agent,
    context=[research_task]
)

# Task 3: Review and enhance the summary
testing_task = Task(
    description="Review the generated summary for structure, clarity, and completeness. Suggest or apply improvements if needed.",
    expected_output="A final polished summary that’s accurate, easy to read, and suitable for AI exam preparation.",
    agent=tester_agent,
    context=[summary_task]
)

##  Run the Crew

The agents now work together to complete the tasks and generate the final summary.


In [ ]:
crew = Crew(tasks=[research_task, summary_task, testing_task])
result = await crew.kickoff_async()

# 6. Format Output
print("\n✅ Final Reviewed Summary:\n",result)

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: AI Research Analyst                                                                                     │
│                                                                                                                 │
│  Task: Research and extract the key concepts, definitions, and examples for the topic: 'Elemental'              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: AI Research Analyst                                                                                     │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Research Note on "Elemental"                                                                                   │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  **1. Definition of Elemental**                                                                                 │
│  - The term "elemental" generally pertains to elements, which are the fundamental substances that cannot be     │
│  broken down into simpler substances by chemical means.                                                         │
│  - In a broad sense, "elemental" describes something that is basic, fundamental, or related to the essential    │
│  components of a system.                                                                                        │
│                                                                                                                 │
│  **2. Core Concepts and Uses of "Elemental"**                                                                   │
│                                                                                                                 │
│  **A. In Chemistry**                                                                                            │
│  - **Elements:** Substances consisting of only one type of atom, characterized by their atomic number.          │
│  - **Elemental Form:** The pure form of an element, as it naturally occurs or in a standard state (e.g.,        │
│  elemental oxygen O₂, elemental gold Au).                                                                       │
│  - **Periodic Table:** The organized chart where elements are arranged based on atomic number, electron         │
│  configurations, and recurring chemical properties.                                                             │
│                                                                                                                 │
│  **B. In Physics**                                                                                              │
│  - Elements are fundamental building blocks of matter, not composed of other substances.                        │
│  - Atomic structure: protons, neutrons, and electrons define each element's identity.                           │
│                                                                                                                 │
│  **C. In Mythology and Fiction**                                                                                │
│  - "Elemental" often refers to beings, forces, or characters embodying one of the classical elements: earth,    │
│  water, fire, air.                                                                                              │
│  - Examples:                                                                                                    │
│    - "Elemental spirits" or "elementals" in fantasy literature and games.                                       │
│    - In popular culture, elemental beings possess powers associated with specific elements.                     │
│                                                        

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Summary Generator                                                                                       │
│                                                                                                                 │
│  Task: Using the research notes, generate a concise exam-style summary with 5–7 indexed bullet points.          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Summary Generator                                                                                       │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  1. "Elemental" primarily refers to chemical elements, which are fundamental substances composed of one type    │
│  of atom, characterized by atomic number, and existing in pure or standard states such as oxygen (O₂) or gold   │
│  (Au).                                                                                                          │
│  2. In chemistry and physics, "elemental" pertains to the basic building blocks of matter, with elements        │
│  defined by atomic structure, and their properties systematically organized in the periodic table.              │
│  3. In mythology and fiction, "elemental" describes beings, forces, or characters embodying the classical       │
│  elements—earth, water, fire, and air—often depicted as powerful primal entities in literature, games, and      │
│  popular culture.                                                                                               │
│  4. The term is also used in art and literature to symbolize primal natural forces and fundamental aspects of   │
│  human experience, often emphasizing themes of nature, power, and elemental energy.                             │
│  5. In technological and scientific contexts, "elemental" relates to analysis techniques (like spectroscopy)    │
│  used to determine the elemental composition of materials and to foundational technologies that support         │
│  complex systems.                                                                                               │
│  6. Examples across contexts include elemental forms of substances like carbon as diamond or graphite, and      │
│  fictional representations such as "The Elementals" or video games centered on elemental forces.                │
│  7. Related topics include periodic trends, allotropes (different structural forms of elements), elemental      │
│  abundance in the universe, and the symbolic importance of elemental forces in mythology and culture.           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Summary Tester                                                                                          │
│                                                                                                                 │
│  Task: Review the generated summary for structure, clarity, and completeness. Suggest or apply improvements if  │
│  needed.                                                                                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Summary Tester                                                                                          │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  "Elemental" primarily refers to chemical elements, which are fundamental substances composed of one type of    │
│  atom characterized by atomic number. These elements exist in pure or standard states, such as oxygen (O₂) or   │
│  gold (Au). In chemistry and physics, "elemental" pertains to the basic building blocks of matter, with         │
│  elements systematically organized in the periodic table based on their atomic structure and properties.        │
│                                                                                                                 │
│  In mythology and fiction, "elemental" describes beings, forces, or characters embodying the classical          │
│  elements—earth, water, fire, and air—often depicted as powerful primal entities in literature, games, and      │
│  popular culture. The term is also used in art and literature to symbolize primal natural forces and            │
│  fundamental aspects of human experience, emphasizing themes of nature, power, and elemental energy.            │
│                                                                                                                 │
│  In technological and scientific contexts, "elemental" relates to analysis techniques such as spectroscopy      │
│  used to determine the elemental composition of materials, as well as foundational technologies that support    │
│  complex systems. Examples across these contexts include elemental forms of substances like carbon (diamond or  │
│  graphite) and fictional representations such as "The Elementals" or video games centered on elemental forces.  │
│                                                                                                                 │
│  Related topics include periodic trends, allotropes (different structural forms of the same element),           │
│  elemental abundance in the universe, and the symbolic significance of elemental forces in mythology and        │
│  culture.                                                                                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


✅ Final Reviewed Summary:
 "Elemental" primarily refers to chemical elements, which are fundamental substances composed of one type of atom characterized by atomic number. These elements exist in pure or standard states, such as oxygen (O₂) or gold (Au). In chemistry and physics, "elemental" pertains to the basic building blocks of matter, with elements systematically organized in the periodic table based on their atomic structure and properties.

In mythology and fiction, "elemental" describes beings, forces, or characters embodying the classical elements—earth, water, fire, and air—often depicted as powerful primal entities in literature, games, and popular culture. The term is also used in art and literature to symbolize primal natural forces and fundamental aspects of human experience, emphasizing themes of nature, power, and elemental energy.

In technological and scientific contexts, "elemental" relates to analysis techniques such as spectroscopy used to determine the elemental 